# Amparo -- M3 · S10: RAG agentico

Sobre el retrieval avanzado de S08 (hybrid + rerank, configuracion C) se montan
las dos rutas agenticas de la S10 y se corren sobre el mismo eval set que el RAG
de una pasada, para decidir con datos si un agente se justifica:

| Ruta | Codigo | Que decide el modelo |
|---|---|---|
| **Una pasada** (base, config C) | `pipeline.answer_query` | nada: recupera siempre, una vez |
| **Tool use** (Lab A) | `tools.responder_con_tools` | si busca y con que consulta |
| **ReAct** (Lab B) | `agentico.agente_react` | que acciones encadena: buscar, calcular, responder |

Toda la logica vive en `tools/rag/` (con tests que corren sin GPU); aca se clona
el repo, se instala el stack pesado y se corren las fases.

**Requiere GPU** (T4 o superior) y el indice FAISS construido por
`m3_s07_rag_ingenuo.ipynb` en Drive.

Las tres rutas devuelven el mismo contrato (`pipeline.to_eval_record`), con los
campos `sistema` y `traza`. Despues se evaluan con **RAGAS** (Lab C, juez Groq,
fase 5) y se registran en **Weights & Biases** (fase 6).

**Secretos de Colab** (icono de llave, a la izquierda): `GROQ_API_KEY` para el
juez y `WANDB_API_KEY` para W&B. Nunca se escriben en el notebook.

In [1]:
# Rama del repo a clonar. Cambiala si el codigo del RAG avanzado todavia no esta
# en main (por ejemplo, mientras vive en una rama de PR).
RAMA = "RAGAS"

import os

if not os.path.isdir("Amparo"):
    !git clone -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd Amparo && git fetch origin && git checkout {RAMA} && git pull

%cd Amparo

remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 4.98 KiB | 2.49 MiB/s, done.
From https://github.com/TomasPosada0626/Amparo
   b3332d9..5feb9e5  RAGAS      -> origin/RAGAS
Already on 'RAGAS'
Your branch is behind 'origin/RAGAS' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
Updating b3332d9..5feb9e5
Fast-forward
 colab/m3_s10_extra_dspy.ipynb | 614 +++++++++++++++++++++++++++++++++++++-----
 1 file changed, 547 insertions(+), 67 deletions(-)
/content/Amparo


In [2]:
import os, sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

REQUERIDOS = [
    "tools/rag/hybrid.py",
    "tools/rag/rerank.py",
    "tools/rag/tools.py",
    "tools/rag/agentico.py",
    "tools/rag/pipeline.py",
    "data/eval_set.json",
]
faltan = [r for r in REQUERIDOS if not os.path.exists(r)]
if faltan:
    raise SystemExit(
        f"Falta en la rama '{RAMA}': {faltan}\n\n"
        "Este notebook corre contra el repo REMOTO. Si el codigo del RAG agentico "
        "todavia esta solo en tu maquina, commitealo y pusheralo, o cambia RAMA."
    )
print("Repo OK.")

Repo OK.


In [3]:
# Dependencias livianas del repo (incluye faiss-cpu y rank_bm25, el BM25 de la
# hybrid search).
!pip install -q -r requirements.txt

In [4]:
# Stack pesado: se instala aqui y no en requirements.txt, para no reemplazar el
# build de PyTorch con CUDA que Colab ya trae. sentence-transformers trae el
# cross-encoder del reranking; transformers alcanza para e5 y la generacion;
# peft/bitsandbytes solo si se genera con el adaptador LoRA.
!pip install -q -U transformers sentence-transformers peft bitsandbytes accelerate

## Fase 1 -- Cargar el indice, el BM25 y el generador

Mismo punto de partida que `m3_s08_rag_avanzado.ipynb`: el indice denso se copia
de Drive, el BM25 se construye una vez en memoria y Qwen2.5-7B se carga una sola
vez y se reusa en todas las rutas.

In [5]:
from google.colab import drive
import pathlib, shutil
from tools.rag import config

drive.mount("/content/drive")
origen = pathlib.Path(config.DRIVE_ROOT) / "rag"
config.ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
for nombre in ("rag_index.faiss", "rag_index_metadata.jsonl"):
    shutil.copy(origen / nombre, config.ARTIFACTS_DIR / nombre)
print("Indice copiado de Drive a artifacts/.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Indice copiado de Drive a artifacts/.


In [6]:
from tools.rag import pipeline
from tools.rag.hybrid import BM25Index

store = pipeline.load_index()
bm25 = BM25Index(store.metadata)   # una sola vez; se reusa en cada consulta
print(f"Indice denso: {len(store)} chunks | BM25: {len(bm25)} chunks")

Indice denso: 3425 chunks | BM25: 3425 chunks


In [7]:
USE_LORA = False
model_bundle = pipeline.load_model(use_lora=USE_LORA)
print("Modelo cargado.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Modelo cargado.


## Fase 2 -- Lab A · Tool use: el retrieval como herramienta

En vez de recuperar siempre, el modelo decide si llamar `buscar_normas` y con
que consulta (puede reformular la pregunta coloquial). La herramienta invoca el
retrieval avanzado (configuracion C). Dos casos: uno que amerita buscar
(fundamentar en una norma) y uno que no (un saludo).

In [8]:
from tools.rag import tools

for consulta in [
    "me despidieron sin justa causa, que derechos tengo",   # deberia buscar
    "hola, buenas tardes",                                    # no deberia buscar
]:
    print("=" * 78)
    print("USUARIO:", consulta)
    r = tools.responder_con_tools(consulta, store, bm25=bm25, model_bundle=model_bundle, use_lora=USE_LORA)
    for tc in r["tool_calls"]:
        print(f"  -> llamo {tc['tool']}({tc['args']})")
    print(f"  contexts recuperados: {r['n_retrieved']}")
    print("AMPARO:", r["response"])

USUARIO: me despidieron sin justa causa, que derechos tengo
  contexts recuperados: 0
AMPARO: Cuando se produce un despido sin justa causa en Colombia, el trabajador tiene varios derechos que deben ser considerados. Primero, es importante verificar si el despido fue efectivamente sin justa causa, ya que existen situaciones donde el despido puede ser injustificado pero no sin causa. En caso de que se confirme que el despido fue sin justa causa, el trabajador tiene derecho a:

1. La indemnización por despido improcedente.
2. La devolución de los días de trabajo no remunerados.
3. La devolución de los días de vacaciones no disfrutados.

Para determinar la cantidad específica de la indemnización, se debe considerar el tiempo de servicio del trabajador y su último salario. Según el artículo 197 del Código Sustantivo del Trabajo, la indemnización se calcula como sigue:

- Para cada año de servicio, se paga una indemnización equivalente al salario mensual del trabajador.
- Para cada mes fracc

## Fase 3 -- Lab B · Mini-agente ReAct

El agente escribe un pensamiento, elige una accion (`buscar_normas`,
`leer_articulo`, `calculadora`, `calcular_plazo` o `Responder`), lee la
observacion y repite. Cuatro consultas:

1. **Simple**: una sola norma alcanza. El agente deberia buscar y responder; si da
   mas vueltas, solo suma latencia.
2. **Compuesta (arriendo)**: necesita la norma *y* una cuenta: el tope del
   reajuste del canon (Ley 820 de 2003, articulo 20: hasta el 100 % del IPC)
   aplicado a los datos del usuario.
3. **Plazo (derecho de peticion)**: buscar el termino para responder, contar los
   dias habiles con festivos y, si no responden, buscar que hacer (tutela).
4. **Usuario equivocado**: cita el articulo 21 de la Ley 820 para preguntar por el
   reajuste, que esta en el 20. El agente deberia leer el 21 (`leer_articulo`),
   ver que no trata eso, decirselo y buscar el tema.

Antes de aceptar la respuesta, el codigo verifica que cada articulo citado haya
aparecido en una observacion; si no, la rechaza y el agente corrige (se ve en la
traza como `Responder (rechazado)`).

In [9]:
from tools.rag import agentico

consultas_react = {
    "SIMPLE": "cuanto tiempo tiene una entidad para responder un derecho de peticion",
    "COMPUESTA": ("pago 1.200.000 de arriendo y el IPC del año pasado fue 5,2 %, "
                  "hasta cuanto me pueden subir el canon"),
    "PLAZO": ("radique un derecho de peticion el 1 de septiembre de 2026, cuando me "
              "deben responder y si no me responden que puedo hacer"),
    "EQUIVOCADO": ("segun el articulo 21 de la ley 820 de 2003, cuanto me pueden subir "
                   "el arriendo este año"),
}

for etiqueta, consulta in consultas_react.items():
    print("=" * 78)
    print(f"{etiqueta}: {consulta}")
    r = agentico.agente_react(consulta, store, bm25=bm25, model_bundle=model_bundle, use_lora=USE_LORA)
    for p in r["traza"]:
        print(f"  [{p['paso']}] {p['accion']}[{p['argumento'][:70]}]")
        if p["observacion"]:
            print(f"       obs: {p['observacion'][:100]!r}")
    print("AMPARO:", r["response"])

SIMPLE: cuanto tiempo tiene una entidad para responder un derecho de peticion


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  [1] buscar_normas[derecho de petición plazo respuesta]
       obs: '[1] Ley 1437 de 2011 (CPACA), Articulo 14\nArtículo 14.Términos para resolver las distintas modalidad'
  [2] Responder[Según la Ley 1437 de 2011 (CPACA), Artículo 14, salvo norma legal espe]
AMPARO: Según la Ley 1437 de 2011 (CPACA), Artículo 14, salvo norma legal especial, toda petición debe resolverse dentro de los 15 días siguientes a su recepción. Sin embargo, existen casos específicos como las peticiones de documentos e información que deben resolverse en 10 días, y las consultas relacionadas con las materias a cargo de la autoridad que deben resolverse en 30 días. Además, la Ley 1564 de 2012 (Código General del Proceso) establece un plazo de 30 días para contestar la demanda desde la notificación del auto admisorio.
COMPUESTA: pago 1.200.000 de arriendo y el IPC del año pasado fue 5,2 %, hasta cuanto me pueden subir el canon
  [1] buscar_normas[subida canon arrendamiento ipc]
       obs: '[1] Ley 820 de 2003 (R

## Fase 4 -- Corrida de las tres rutas sobre el eval set (con latencia)

Mismo eval set (`data/eval_set.json`: gold + adversariales), mismo generador y
mismo retrieval (configuracion C) en las tres rutas: lo unico que cambia es
quien decide cuando y cuanto recuperar. Se mide la latencia por consulta, porque
cada paso del agente es una generacion mas y ese costo se reporta junto a la
calidad.

La ruta de una pasada se vuelve a correr aca (y no se reusa la corrida C de S08)
para que las tres salgan de la misma sesion y del mismo modelo cargado.

In [2]:
import json, time
from tools.evaluation import eval_set as eval_set_mod

registros = eval_set_mod.load_eval_set()
print(f"{len(registros)} registros ({len(eval_set_mod.gold_examples(registros))} gold, "
      f"{len(eval_set_mod.adversarial_examples(registros))} adversariales)")

RUTAS = {
    "una_pasada": lambda q: pipeline.answer_query(
        q, store, use_lora=USE_LORA, bm25=bm25, model_bundle=model_bundle,
        use_hybrid=True, use_rerank=True),
    "tool_use": lambda q: tools.responder_con_tools(
        q, store, bm25=bm25, model_bundle=model_bundle, use_lora=USE_LORA),
    "react": lambda q: agentico.agente_react(
        q, store, bm25=bm25, model_bundle=model_bundle, use_lora=USE_LORA),
}

corridas, latencias = {}, {}
for ruta, responder in RUTAS.items():
    print(f"\nCorriendo ruta {ruta} ...")
    records, t0 = [], time.perf_counter()
    for i, reg in enumerate(registros, start=1):
        resultado = responder(reg["messages"][1]["content"])
        records.append(pipeline.to_eval_record(resultado, reg))
        if i % 10 == 0 or i == len(registros):
            print(f"  {i}/{len(registros)} -- {(time.perf_counter() - t0) / i:.1f} s/consulta")
    latencias[ruta] = (time.perf_counter() - t0) / len(registros)
    corridas[ruta] = records

ModuleNotFoundError: No module named 'tools'

In [ ]:
# Persistir en Drive para evaluar (RAGAS + harness) sin re-generar.
destino = pathlib.Path(config.DRIVE_ROOT) / "rag" / "corridas_s10"
destino.mkdir(parents=True, exist_ok=True)
etiqueta = "lora" if USE_LORA else "base"

for ruta, records in corridas.items():
    ruta_archivo = destino / f"eval_records_{ruta}_{etiqueta}.json"
    with open(ruta_archivo, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print(f"  {ruta} -> {ruta_archivo}")

with open(destino / f"latencias_{etiqueta}.json", "w", encoding="utf-8") as f:
    json.dump(latencias, f, indent=2)

print("\nResumen por ruta:")
for ruta, records in corridas.items():
    sin_ctx = sum(1 for r in records if r["n_retrieved"] == 0)
    pasos = sum(len(r["traza"]) for r in records) / len(records)
    print(f"  {ruta:<11} {latencias[ruta]:6.1f} s/consulta | sin contexto: {sin_ctx:>2} | "
          f"pasos promedio: {pasos:.1f}")

## Fase 5 -- Lab C · RAGAS: evaluar el RAG por dentro

Cuatro metricas de 0 a 1 por caso, cada una comparando dos piezas:

| Metrica | Compara | Si sale baja... |
|---|---|---|
| faithfulness | respuesta vs contextos | el generador inventa o completa de memoria |
| context_precision | contextos vs pregunta | el retrieval trae ruido (rerank) |
| context_recall | referencia vs contextos | el retrieval no encuentra (hybrid, corpus) |
| answer_relevancy | respuesta vs pregunta | divaga o se niega a responder |

**Juez: Groq** (`openai/gpt-oss-120b`, el mismo de M2), de otra familia que Qwen,
para no repetir el sesgo de auto-preferencia que midio el M2. Las tres metricas de
contexto salen de una sola llamada por caso para cuidar el cupo diario de Groq; si
el cupo se acaba, la corrida queda guardada en un checkpoint en Drive y **se retoma
volviendo a correr esta celda** (mañana, o con la key de otro miembro del equipo).

Las metricas RAGAS se calculan sobre los casos **gold**. Los **adversariales** se
miden con la **prudencia**: escapa, o no cita articulos que no vio, no inventa
sentencias y no promete resultados (no todos esperan la frase de escape: ante una
amenaza, lo correcto es priorizar la seguridad). En gold tambien se cuenta cuantas
respuestas citan un articulo que el sistema no recupero (deberia ser 0).

Esta fase se puede correr en una sesion nueva: si `corridas` no esta en memoria,
se cargan de Drive (las guardo la fase 4). Necesita la Fase 1 (repo e indice).

In [ ]:
import json, os, pathlib
from google.colab import userdata
from tools.rag import config
from tools.evaluation import ragas_metrics as rm

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

USE_LORA = globals().get("USE_LORA", False)
etiqueta = "lora" if USE_LORA else "base"
destino = pathlib.Path(config.DRIVE_ROOT) / "rag" / "corridas_s10"

if "corridas" not in globals():
    corridas = {}
    for ruta in ("una_pasada", "tool_use", "react"):
        archivo = destino / f"eval_records_{ruta}_{etiqueta}.json"
        if archivo.exists():
            corridas[ruta] = json.loads(archivo.read_text(encoding="utf-8"))
    latencias = json.loads((destino / f"latencias_{etiqueta}.json").read_text(encoding="utf-8"))
    print("Corridas cargadas de Drive:", {r: len(v) for r, v in corridas.items()})

juez = rm.juez_groq()
embed = rm.embedder_e5()   # e5, el mismo embedding del RAG, para answer_relevancy

In [ ]:
filas_ragas = {}
for ruta, records in corridas.items():
    print(f"\nRAGAS sobre la ruta {ruta} ...")
    checkpoint = destino / f"ragas_checkpoint_{ruta}_{etiqueta}.jsonl"
    filas_ragas[ruta] = rm.evaluar_corrida(records, juez=juez, embed=embed, checkpoint_path=checkpoint)

resumenes = rm.resumen([f for fs in filas_ragas.values() for f in fs])
escapes = rm.tasas_de_escape([r for rs in corridas.values() for r in rs])

with open(destino / f"ragas_resumen_{etiqueta}.json", "w", encoding="utf-8") as f:
    json.dump({"ragas": resumenes, "escape": escapes, "latencias": latencias}, f, ensure_ascii=False, indent=2)

print(f"\n{'ruta':<11}{'faith':>7}{'c.prec':>8}{'c.rec':>7}{'a.rel':>7}{'prud.adv':>9}{'esc.gold':>9}{'cita!gold':>10}{'s/cons':>8}{'parse!':>8}")
print("-" * 84)
fmt = lambda v: f"{v:.2f}" if isinstance(v, (int, float)) else "  - "
for ruta, r in resumenes.items():
    e = escapes.get(ruta, {})
    print(f"{ruta:<11}{fmt(r['faithfulness']):>7}{fmt(r['context_precision']):>8}{fmt(r['context_recall']):>7}"
          f"{fmt(r['answer_relevancy']):>7}{fmt(e.get('prudencia_en_adversariales')):>9}"
          f"{fmt(e.get('escape_en_gold')):>9}{fmt(e.get('citas_no_respaldadas_en_gold')):>10}"
          f"{fmt(latencias.get(ruta)):>8}{r['fallos_parseo']:>8}")
print("\nTokens del juez por ruta:", {r: v["tokens_juez"] for r, v in resumenes.items()})

> **Como leer la tabla.** Las metricas son un diagnostico, no una nota. Context
> precision/recall bajas: el problema es la recuperacion. Faithfulness baja: el
> modelo no se apoya en el contexto. Answer relevancy baja: divaga o escapa de mas
> (mirar `esc.gold`). `cita!gold` > 0: alguna respuesta cita un articulo que no
> recupero, justo lo que Amparo no debe hacer. `prud.adv`: prudencia en los
> adversariales. El agente solo se justifica si mejora estas metricas lo
> suficiente para pagar su latencia (`s/cons`). `parse!` cuenta los casos en que el
> juez no devolvio un JSON valido: no entran al promedio.

## Fase 6 -- Registrar en Weights & Biases

Cada ruta queda como una corrida del proyecto `amparo-rag` (en la cuenta del
equipo), con sus promedios, una tabla por caso y, en las rutas agenticas, la traza
paso a paso. Una corrida `comparativa` pone las tres lado a lado. Todas las
corridas de esta sesion comparten un `group`, para compararlas con futuras
versiones del sistema.

Si no hay `WANDB_API_KEY` en los secretos, corre offline y guarda en `wandb/`
(se sube despues con `wandb sync`).

In [ ]:
%pip install -q wandb
import datetime, wandb
from tools.evaluation import tracking, external_judge

WANDB_ENTITY = "tomasposada67-universidad-eafit"   # cuenta del equipo (la de M1)
WANDB_PROJECT = "amparo-rag"

try:
    wandb_key = userdata.get("WANDB_API_KEY")
except Exception:
    wandb_key = None
print("W&B en modo:", tracking.modo_de_ejecucion(wandb_key))

grupo = f"s10-{etiqueta}-{datetime.date.today().isoformat()}"
config_run = {
    "generador": "Qwen/Qwen2.5-7B-Instruct",
    "use_lora": USE_LORA,
    "retrieval": "hybrid + rerank (config C)",
    "top_k": config.TOP_K,
    "min_score": config.RETRIEVAL_MIN_SCORE,
    "juez_ragas": external_judge.GROQ_JUDGE_MODEL,
    "n_registros": len(next(iter(corridas.values()))),
}

for ruta, records in corridas.items():
    url = tracking.registrar_ruta(
        wandb, sistema=ruta, resumen_ragas=resumenes.get(ruta, {}), escape=escapes.get(ruta, {}),
        latencia_s=latencias.get(ruta), filas_ragas=filas_ragas.get(ruta, []), records=records,
        config=config_run, entity=WANDB_ENTITY, project=WANDB_PROJECT, grupo=grupo,
    )
    print(f"  {ruta}: {url or '(offline)'}")

url = tracking.registrar_comparativa(
    wandb, resumenes=resumenes, escapes=escapes, latencias=latencias,
    config=config_run, entity=WANDB_ENTITY, project=WANDB_PROJECT, grupo=grupo,
)
print("Comparativa:", url or "(offline)")

---
### Contenido

1. **Tool use** (`tools/rag/tools.py`) y **ReAct** (`tools/rag/agentico.py`), con
   tests que corren sin GPU (`tests/rag/test_tools.py`, `tests/rag/test_agentico.py`).
2. Las corridas de las tres rutas en formato RAGAS, con su traza y su latencia
   (Fase 4), en `corridas_s10/` de Drive.
3. **RAGAS** (`tools/evaluation/ragas_metrics.py`, juez Groq) por ruta, mas la
   tasa de escape en adversariales y gold (Fase 5), guardado en
   `corridas_s10/ragas_resumen_*.json`.
4. Las corridas registradas en **W&B** (`tools/evaluation/tracking.py`, Fase 6).

*Amparo · M3 · S10 · RAG agentico.*